<a href="https://colab.research.google.com/github/Aki55755/Applications-of-data-mining/blob/main/python_assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!apt-get install flex bison gcc -y

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

from PIL import Image
from io import BytesIO

# ============================================================
# 1. SETTINGS
# ============================================================

IMG_SIZE = 224

IMAGE_URLS = {
    "grace_hopper.jpg":
        "https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg",

    "grizzly_bear.jpg":
        "https://storage.googleapis.com/download.tensorflow.org/example_images/592px-Young_Grizzly_Bear.jpg",

    "cat.jpg":
        "https://storage.googleapis.com/download.tensorflow.org/example_images/320px-Felis_catus-cat_on_snow.jpg"
}

# ============================================================
# 2. DOWNLOAD IMAGES DIRECTLY
# ============================================================

os.makedirs("test_images", exist_ok=True)

image_paths = []

for filename, url in IMAGE_URLS.items():

    path = os.path.join("test_images", filename)

    print("Downloading:", filename)

    response = requests.get(url)

    if response.status_code == 200:
        with open(path, "wb") as f:
            f.write(response.content)

        image_paths.append(path)
        print("Downloaded successfully")

    else:
        print("Download failed:", filename)


# ============================================================
# 3. LOAD IMAGE
# ============================================================

def load_image(path):

    image = Image.open(path).convert("RGB")
    image = image.resize((IMG_SIZE, IMG_SIZE))

    image_array = np.array(image)
    image_array = np.expand_dims(image_array, axis=0)

    return image_array


# ============================================================
# 4. MODEL 1 - CONVENTIONAL CNN
#    ResNet50
# ============================================================

def create_resnet50():

    model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=True
    )

    return model


# ============================================================
# 5. MODEL 2 - ATTENTION-ENHANCED CNN
#    ResNet50 + Channel Attention
# ============================================================

def attention_block(x):

    channels = x.shape[-1]

    attention = tf.keras.layers.GlobalAveragePooling2D()(x)

    attention = tf.keras.layers.Dense(
        channels // 16,
        activation="relu"
    )(attention)

    attention = tf.keras.layers.Dense(
        channels,
        activation="sigmoid"
    )(attention)

    attention = tf.keras.layers.Reshape(
        (1, 1, channels)
    )(attention)

    x = tf.keras.layers.Multiply()([x, attention])

    return x


def create_attention_cnn():

    base = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    x = base.output

    x = attention_block(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(
        1000,
        activation="softmax"
    )(x)

    model = tf.keras.Model(
        inputs=base.input,
        outputs=x
    )

    return model


# ============================================================
# 6. MODEL 3 - MODERN CNN
#    EfficientNetV2B0
# ============================================================

def create_efficientnet():

    model = tf.keras.applications.EfficientNetV2B0(
        weights="imagenet",
        include_top=True
    )

    return model


# ============================================================
# 7. MODEL 4 - LIGHTWEIGHT CNN
#    MobileNetV2
# ============================================================

def create_mobilenet():

    model = tf.keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=True
    )

    return model


# ============================================================
# 8. MODEL 5 - VISION TRANSFORMER
#    Simple ViT implementation
# ============================================================

class Patches(tf.keras.layers.Layer):

    def __init__(self, patch_size):

        super().__init__()

        self.patch_size = patch_size

    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],
            strides=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [
                batch_size,
                -1,
                patch_dims
            ]
        )

        return patches


class PatchEncoder(tf.keras.layers.Layer):

    def __init__(self, num_patches, projection_dim):

        super().__init__()

        self.projection = tf.keras.layers.Dense(
            units=projection_dim
        )

        self.position_embedding = tf.keras.layers.Embedding(
            input_dim=num_patches,
            output_dim=projection_dim
        )

    def call(self, patches):

        positions = tf.range(
            start=0,
            limit=tf.shape(patches)[1],
            delta=1
        )

        encoded = self.projection(patches)

        encoded += self.position_embedding(positions)

        return encoded


def create_vit():

    image_size = 224

    patch_size = 16

    projection_dim = 64

    num_heads = 4

    transformer_layers = 4

    num_patches = (image_size // patch_size) ** 2

    inputs = tf.keras.Input(
        shape=(image_size, image_size, 3)
    )

    # Normalize
    x = tf.keras.layers.Rescaling(
        1.0 / 255
    )(inputs)

    # Create patches
    patches = Patches(patch_size)(x)

    # Encode patches
    encoded = PatchEncoder(
        num_patches,
        projection_dim
    )(patches)

    # Transformer blocks
    for _ in range(transformer_layers):

        x1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )(encoded)

        attention_output = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim
        )(
            x1,
            x1
        )

        x2 = tf.keras.layers.Add()(
            [attention_output, encoded]
        )

        x3 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )(x2)

        x3 = tf.keras.layers.Dense(
            projection_dim * 2,
            activation=tf.nn.gelu
        )(x3)

        x3 = tf.keras.layers.Dense(
            projection_dim
        )(x3)

        encoded = tf.keras.layers.Add()(
            [x3, x2]
        )

    representation = tf.keras.layers.LayerNormalization(
        epsilon=1e-6
    )(encoded)

    representation = tf.keras.layers.GlobalAveragePooling1D()(
        representation
    )

    representation = tf.keras.layers.Dropout(0.1)(
        representation
    )

    outputs = tf.keras.layers.Dense(
        1000,
        activation="softmax"
    )(representation)

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs
    )

    return model


# ============================================================
# 9. CREATE MODELS
# ============================================================

models = {

    "Conventional CNN - ResNet50":
        create_resnet50(),

    "Attention CNN - ResNet50 + Attention":
        create_attention_cnn(),

    "Modern CNN - EfficientNetV2B0":
        create_efficientnet(),

    "Lightweight CNN - MobileNetV2":
        create_mobilenet(),

    "Vision Transformer - ViT":
        create_vit()
}


# ============================================================
# 10. LOAD IMAGENET LABELS
# ============================================================

labels_url = (
    "https://storage.googleapis.com/"
    "download.tensorflow.org/data/"
    "imagenet_class_index.json"
)

labels_response = requests.get(labels_url)

imagenet_labels = labels_response.json()

labels = {
    int(k): v[1]
    for k, v in imagenet_labels.items()
}


# ============================================================
# 11. EVALUATE MODELS
# ============================================================

results = []

for model_name, model in models.items():

    print("\n======================================")
    print(model_name)
    print("======================================")

    total_time = 0

    correct = 0

    total = 0

    for image_path in image_paths:

        image = load_image(image_path)

        start = time.time()

        prediction = model.predict(
            image,
            verbose=0
        )

        inference_time = time.time() - start

        total_time += inference_time

        predicted_class = np.argmax(
            prediction[0]
        )

        confidence = prediction[0][predicted_class]

        predicted_label = labels.get(
            predicted_class,
            "Unknown"
        )

        print(
            os.path.basename(image_path),
            "->",
            predicted_label,
            f"({confidence * 100:.2f}%)",
            f"Time: {inference_time:.4f}s"
        )

        total += 1

    parameter_count = model.count_params()

    average_time = total_time / len(image_paths)

    results.append({

        "Model": model_name,

        "Parameters":
            parameter_count,

        "Average Inference Time (sec)":
            average_time

    })


# ============================================================
# 12. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)

print("\n\n================ FINAL COMPARISON ================\n")

print(
    results_df.to_string(
        index=False
    )
)

# Save results
results_df.to_csv(
    "cnn_vit_comparison.csv",
    index=False
)

print(
    "\nResults saved as cnn_vit_comparison.csv"
)

Downloading: grace_hopper.jpg
Downloaded successfully
Downloading: grizzly_bear.jpg
Download failed: grizzly_bear.jpg
Downloading: cat.jpg
Downloaded successfully
102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
29403144/29403144 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Conventional CNN - ResNet50
grace_hopper.jpg -> bow_tie (83.34%) Time: 1.9313s
cat.jpg -> Egyptian_cat (85.71%) Time: 0.3521s

Attention CNN - ResNet50 + Attention
grace_hopper.jpg -> picket_fence (0.67%) Time: 1.8837s
cat.jpg -> dung_beetle (0.53%) Time: 0.2196s

Modern CNN - EfficientNetV2B0
grace_hopper.jpg -> military_uniform (76.93%) Time: 2.2630s
cat.jpg -> lynx (50.09%) Time: 0.1685s

Lightweight CNN - MobileNetV2
grace_hopper.jpg -> pillow (49.27%) Time: 1.6144s
cat.jpg -> shower_curtain (23.35%) Time: 0.1709s

Vision Transformer - ViT


grace_hopper.jpg -> boxer (0.23%) Time: 0.8754s
cat.jpg -> jersey (0.25%) Time: 0.0842s


================ FINAL COMPARISON ================

                               Model  Parameters  Average Inference Time (sec)
         Conventional CNN - ResNet50    25636712                      1.141694
Attention CNN - ResNet50 + Attention    26163176                      1.051673
       Modern CNN - EfficientNetV2B0     7200312                      1.215739
       Lightweight CNN - MobileNetV2     3538984                      0.892662
            Vision Transformer - ViT      459688                      0.479793

Results saved as cnn_vit_comparison.csv


In [ ]:
!flex vamshi.l

/bin/bash: line 1: flex: command not found


In [ ]:
!gcc lex.yy.c -o lexprog

cc1: fatal error: lex.yy.c: No such file or directory
compilation terminated.


In [ ]:
!flex vamshi.l

/bin/bash: line 1: flex: command not found


In [ ]:
!gcc lex.yy.c -o lexprog

cc1: fatal error: lex.yy.c: No such file or directory
compilation terminated.


In [ ]:
!gcc lex.yy.c -o lexprog

cc1: fatal error: lex.yy.c: No such file or directory
compilation terminated.


In [ ]:
!./lexprog

/bin/bash: line 1: ./lexprog: No such file or directory


In [ ]:
!apt_get update

/bin/bash: line 1: apt_get: command not found


In [ ]:
!apt_get install flex bison gcc -y

/bin/bash: line 1: apt_get: command not found
